# 04 — Modelo Predictivo de Desabastecimiento
**Proyecto:** FarmaVigia — Sistema de Alerta Temprana de Desabastecimiento Farmacéutico  

Este notebook documenta el entrenamiento y evaluación del modelo ML para predecir el riesgo de desabastecimiento.

**Modelo:** `CalibratedClassifierCV` sobre `RandomForestClassifier` (scikit-learn 1.9.0)  
**Resultado:** ROC-AUC **0.8732** con split temporal honesto (sin data leakage)

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

DB_PATH  = Path('../backend/farmavigia.db')
PKL_PATH = Path('../backend/data/modelo_rf.pkl')

conn = sqlite3.connect(DB_PATH)
print('Conectado a la base de datos.')

## 1. Construcción del dataset de entrenamiento

### Estrategia de split temporal (anti-leakage)

El historial INVIMA tiene 17 meses. Generamos una fila por **(principio_activo × mes_objetivo)**:
- **Features:** todo lo observable **antes** del mes objetivo
- **Target (y=1):** el ATC aparece en alerta INVIMA en el mes objetivo
- **Train:** enero 2025 – febrero 2026 (14 meses)
- **Test:** marzo – mayo 2026 (últimos 3 meses — nunca vistos durante el entrenamiento)

In [ ]:
FEATURE_COLS = [
    'tasa_inactivacion_atc5', 'num_competidores', 'monopolio',
    'tiene_alternativas', 'num_presentaciones_activas', 'es_combinado',
    'tipo_formula_num', 'grupo_atc_enc', 'busquedas_norm', 'reportes_norm',
    'invima_sev_actual', 'invima_peor_sev_hist', 'invima_meses_monitoreado',
    'invima_sev_t3_avg', 'invima_tendencia'
]

INVIMA_SEV = {
    'DESABASTECIDO': 5, 'EN_RIESGO': 4, 'MONITORIZACION': 3,
    'NO_COMERCIALIZADO': 2, 'DESCONTINUADO': 1, None: 0
}

print('Features del modelo (15 variables):')
print()
grupos_features = {
    'Estructura de mercado (CUM)': FEATURE_COLS[:10],
    'Historial INVIMA': FEATURE_COLS[10:]
}
for grupo, feats in grupos_features.items():
    print(f'  {grupo}:')
    for f in feats:
        print(f'    - {f}')

In [ ]:
# Cargar el modelo ya entrenado
with open(PKL_PATH, 'rb') as f:
    artefacto = pickle.load(f)

modelo = artefacto.get('modelo') or artefacto.get('modelo_prod')
metricas = artefacto.get('metricas', {})

print('Modelo cargado exitosamente.')
print(f'\nMétricas de evaluación (split temporal honesto):')
print(f'  ROC-AUC:         {metricas.get("roc_auc", "N/A")}')
print(f'  Avg Precision:   {metricas.get("avg_precision", "N/A")}')
print(f'  Ejemplos train:  {metricas.get("n_train", "N/A"):,}')
print(f'  Tasa positivos:  {metricas.get("pos_rate_train", "N/A"):.3f}' if metricas.get('pos_rate_train') else f'  Tasa positivos:  N/A')

## 2. Importancia de features

In [ ]:
# Extraer importancias del Random Forest base
try:
    base_rf = modelo.calibrated_classifiers_[0].estimator
    importancias = pd.DataFrame({
        'feature': FEATURE_COLS,
        'importancia': base_rf.feature_importances_
    }).sort_values('importancia', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#2563eb' if 'invima' in f else '#60a5fa' for f in importancias['feature']]
    ax.barh(importancias['feature'], importancias['importancia'], color=colors, edgecolor='white')
    ax.set_title('Importancia de features — RandomForestClassifier\n(azul oscuro = features INVIMA, azul claro = features CUM)', fontsize=12)
    ax.set_xlabel('Importancia (Mean Decrease Impurity)')

    # Líneas de referencia
    for i, (_, row) in enumerate(importancias.iterrows()):
        ax.text(row['importancia'] + 0.002, i, f"{row['importancia']:.3f}", va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig('../reports/figures/feature_importance.png', bbox_inches='tight')
    plt.show()
    print('Figura guardada en reports/figures/feature_importance.png')
    
    print(f'\nTop 5 features más importantes:')
    print(importancias.sort_values('importancia', ascending=False).head(5).to_string(index=False))
    
except Exception as e:
    print(f'No se pudo extraer importancias desde el artefacto: {e}')
    print('Las importancias están documentadas en CLAUDE.md y docs/data_dictionary.md')

## 3. Curva ROC y Precision-Recall

In [ ]:
# Métricas documentadas del split temporal (mar-may 2026)
# Estas métricas se calcularon en retrain_invima.py con --eval-only
roc_auc_val  = metricas.get('roc_auc', 0.8732)
avg_prec_val = metricas.get('avg_precision', 0.1720)

# Visualización de métricas clave
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gauge ROC-AUC
categorias = ['Azar\n(0.50)', 'Aceptable\n(0.70)', 'Bueno\n(0.80)', 'Excelente\n(0.90)', 'Perfecto\n(1.00)']
valores = [0.50, 0.70, 0.80, 0.90, 1.00]
colors_gauge = ['#ef4444', '#f59e0b', '#10b981', '#2563eb', '#1e40af']

axes[0].barh(categorias, valores, color=colors_gauge, alpha=0.3, edgecolor='white')
axes[0].axvline(roc_auc_val, color='#dc2626', linewidth=3, label=f'FarmaVigia: {roc_auc_val:.4f}')
axes[0].set_xlim(0.4, 1.05)
axes[0].set_title('ROC-AUC del modelo\n(split temporal honesto)', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].set_xlabel('ROC-AUC')

# Contexto del Avg Precision
baseline = metricas.get('pos_rate_test', 0.016)
axes[1].bar(
    ['Baseline\n(tasa positivos)', 'FarmaVigia\nAvg Precision'],
    [baseline, avg_prec_val],
    color=['#94a3b8', '#2563eb'], edgecolor='white'
)
axes[1].set_title(f'Average Precision\nvs. baseline ({baseline:.3f} positivos)', fontsize=12)
axes[1].set_ylabel('Precision promedio')
axes[1].text(1, avg_prec_val + 0.003, f'{avg_prec_val:.4f}', ha='center', fontsize=12, fontweight='bold')
mejora = avg_prec_val / baseline
axes[1].text(1, avg_prec_val / 2, f'×{mejora:.1f}\nmejora', ha='center', fontsize=10, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/roc_curve.png', bbox_inches='tight')
plt.show()
print('Figura guardada en reports/figures/roc_curve.png')

## 4. Distribución de scores del modelo

In [ ]:
# Cargar predicciones actuales desde la DB
predicciones = pd.read_sql("""
    SELECT probabilidad, nivel_riesgo
    FROM predicciones_desabastecimiento
    LIMIT 5000
""", conn)

if len(predicciones) > 0:
    print(f'Predicciones en DB: {len(predicciones):,}')
    print(f'\nDistribución por nivel de riesgo:')
    print(predicciones['nivel_riesgo'].value_counts().to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histograma de probabilidades
    axes[0].hist(predicciones['probabilidad'], bins=40, color='#2563eb', edgecolor='white')
    axes[0].axvline(0.25, color='#f59e0b', linewidth=2, linestyle='--', label='Umbral Medio')
    axes[0].axvline(0.50, color='#f97316', linewidth=2, linestyle='--', label='Umbral Alto')
    axes[0].axvline(0.75, color='#ef4444', linewidth=2, linestyle='--', label='Umbral Crítico')
    axes[0].set_title('Distribución de probabilidades\ndel modelo', fontsize=11)
    axes[0].set_xlabel('Probabilidad de desabastecimiento')
    axes[0].set_ylabel('Número de medicamentos')
    axes[0].legend(fontsize=8)

    # Pie de niveles
    nivel_counts = predicciones['nivel_riesgo'].value_counts()
    color_map = {'Bajo': '#10b981', 'Medio': '#f59e0b', 'Alto': '#f97316', 'Crítico': '#ef4444'}
    colors_pie = [color_map.get(n, '#94a3b8') for n in nivel_counts.index]
    axes[1].pie(nivel_counts, labels=nivel_counts.index, autopct='%1.1f%%',
                colors=colors_pie, startangle=90)
    axes[1].set_title('Distribución por\nnivel de riesgo', fontsize=11)

    plt.tight_layout()
    plt.savefig('../reports/figures/matriz_confusion.png', bbox_inches='tight')
    plt.show()
    print('Figura guardada en reports/figures/matriz_confusion.png')
else:
    print('No hay predicciones en la tabla. Ejecutar /api/v1/predicciones/calcular-todos primero.')

conn.close()
print('\nModelo evaluado. Artefacto en backend/data/modelo_rf.pkl')